In [1]:
import math, os
from time import time
import torch
from DeepTracing import DEEPTRACING
import numpy as np
from pathlib import Path
import scanpy as sc
from preprocess import normalize, inducing_points_select, barcode_distances_between_arrays

torch.manual_seed(0)

In [2]:
class Args(object):
    def __init__(self):
        self.data_file = './data'
        self.select_genes = 0
        self.batch_size = "auto"
        self.maxiter = 5000
        self.train_size = 0.95
        self.patience = 20
        self.lr = 1e-3
        self.weight_decay = 1e-6
        self.noise = 0
        self.dropoutE = 0
        self.dropoutD = 0
        self.encoder_layers = [128, 64]
        self.GP_dim = 5
        self.Normal_dim = 5
        self.decoder_layers = [128]
        self.dynamicVAE = True
        self.init_beta_gaussian = 5
        self.min_beta_gaussian = 1
        self.max_beta_gaussian = 10
        self.init_beta_gp = 10
        self.min_beta_gp = 4
        self.max_beta_gp = 25
        self.KL_loss = 0.025
        self.num_samples = 1
        self.fix_inducing_points = True
        self.inducing_point_nums = None
        self.fixed_gp_params = False
        self.loc_range = 20.
        self.kernel_scale = 20.
        self.denoised_counts_file = "denoised_counts.txt"
        self.num_denoise_samples = 10000
        self.device = "cuda"

args = Args()

In [3]:
p_a = 0.2
output_dir = Path(f"./results/p_a_{p_a}")
output_dir.mkdir(parents=True, exist_ok=True)
model_file = output_dir / f"model.pt"
final_latent_file = output_dir / f"final_latent.txt"
gp_latent_file = output_dir / f"gp_latent.txt"
gaussian_latent_file = output_dir / f"gaussian_latent.txt"

In [4]:
adata = sc.read(Path(args.data_file) / f"preprocessed_adata_{p_a}.h5ad")
print(adata)

# sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=4000, subset=True)
adata = normalize(adata,
                  size_factors=True,
                  normalize_input=True,
                  logtrans_input=True)

AnnData object with n_obs × n_vars = 4096 × 500
    obs: 'time', 'node_depth', 'cluster', 'subtree'
    obsm: 'barcodes'


/data/data2/wenyuhong/miniconda3/envs/DeepTracing/lib/python3.9/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [5]:
barcodes_all = np.asarray(adata.obsm["barcodes"])  # (N, 32)
unique_barcodes, inverse_indices = np.unique(barcodes_all, axis=0, return_inverse=True)

indices_all = inverse_indices

inducing_points = inducing_points_select(adata=adata, barcodes=barcodes_all,
                                     select_feature=adata.obs['subtree'],
                                     all_indices=inverse_indices, CELLS_PER_ADDITIONAL_POINT=100)
initial_inducing_points = np.array(inducing_points)
print(initial_inducing_points)
print(f"total number of induction points: {len(initial_inducing_points)}")

[2287 1698 1785 2085 1284 2353 1772 1238 1079  615  581 1071 1437 1411
 1508 1521  517  639  752  631  560  663  749 1716 1327  502 1353  758
  725 1856 1347  915  252  392    0   48 2612 2521  375  423 3644  273
 2179  180  126  154  111 1826 3009 3010 3295 3089 3486 2969 3185 3503
 2914 2844 2827 3355 2761 3528 3282 2729]
total number of induction points: 64


In [6]:
distance_matrix_file = output_dir / f"unique_distance_matrix_{p_a}.npy"
if not os.path.isfile(distance_matrix_file):
    unique_distance_matrix = barcode_distances_between_arrays(unique_barcodes, unique_barcodes)
    np.save(distance_matrix_file, unique_distance_matrix)
unique_distance_matrix = np.load(distance_matrix_file)

In [7]:
if args.batch_size == "auto":
    if adata.X.shape[0] <= 1024:
        args.batch_size = 128
    elif adata.X.shape[0] <= 2048:
        args.batch_size = 256
    else:
        args.batch_size = 512
else:
    args.batch_size = int(args.batch_size)
print(args)

In [8]:
model = DEEPTRACING(input_dim=adata.n_vars, GP_dim=args.GP_dim, Normal_dim=args.Normal_dim, encoder_layers=args.encoder_layers, decoder_layers=args.decoder_layers,
        noise=args.noise, encoder_dropout=args.dropoutE, decoder_dropout=args.dropoutD, distance=unique_distance_matrix,
        initial_inducing_points=initial_inducing_points, fixed_inducing_points=args.fix_inducing_points,
        fixed_gp_params=args.fixed_gp_params, kernel_scale=args.kernel_scale, N_train=adata.n_obs, KL_loss=args.KL_loss, dynamicVAE=args.dynamicVAE,
        init_beta_gaussian=args.init_beta_gaussian, min_beta_gaussian=args.min_beta_gaussian, max_beta_gaussian=args.max_beta_gaussian,
        init_beta_gp=args.init_beta_gp, min_beta_gp=args.min_beta_gp, max_beta_gp=args.max_beta_gp,
        dtype=torch.float64, device=args.device)

print(str(model))

DEEPTRACING(
  (svgp): SVGP(
    (kernel): Kernel()
  )
  (encoder): DenseEncoder(
    (layers): Sequential(
      (0): Linear(in_features=500, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ELU(alpha=1.0)
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ELU(alpha=1.0)
    )
    (enc_mu): Linear(in_features=64, out_features=10, bias=True)
    (enc_var): Linear(in_features=64, out_features=10, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=10, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
  )
  (dec_mean): Sequential(
    (0): Linear(in_features=128, out_features=500, bias=True)
    (1): MeanAct()
  )
  (NB_loss): NBLoss()
)


In [9]:
if not model_file.exists():
# if True:
    t0 = time()
    model.train_model(indices=indices_all, ncounts=adata.X, raw_counts=adata.raw.X, size_factors=adata.obs["size_factors"].to_numpy(),
            lr=args.lr, weight_decay=args.weight_decay, batch_size=args.batch_size, num_samples=args.num_samples,
            train_size=args.train_size, maxiter=args.maxiter, patience=args.patience, save_model=True, model_weights=str(model_file))
    print('Training time: %d seconds.' % int(time() - t0))
else:
    model.load_model(str(model_file))

# model.curve_loss()

Training


/data/data2/wenyuhong/DeepTracing/example/VAE_utils.py:75: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /data/data2/wenyuhong/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at  /opt/conda/conda-bld/pytorch_1646756402876/work/aten/src/ATen/native/cuda/jit_utils.cpp:860.)
  t1 = torch.lgamma(disp+eps) + torch.lgamma(x+1.0) - torch.lgamma(x+disp+eps)


Training epoch 1, ELBO:6014.73395060, NB loss:6002.51304331, GP KLD loss:0.05767796, Gaussian KLD loss:2.28181533, noise regularization:0.000000
Current beta_gaussian 5.234197420040332
Current beta_gp 10.234197420040331
Current kernel scale tensor([20.0031], device='cuda:0')
Training epoch 1, validating ELBO:5455.52610040
Training epoch 2, ELBO:5202.03343895, NB loss:5175.16953594, GP KLD loss:0.10024688, Gaussian KLD loss:4.73271890, noise regularization:0.000000
Current beta_gaussian 5.783110057369769
Current beta_gp 10.783110057369768
Current kernel scale tensor([20.0004], device='cuda:0')
Training epoch 2, validating ELBO:4784.83656487
Training epoch 3, ELBO:4584.22112966, NB loss:4542.28332803, GP KLD loss:0.13274272, Gaussian KLD loss:6.55974784, noise regularization:0.000000
Current beta_gaussian 6.704699034805087
Current beta_gp 11.704699034805085
Current kernel scale tensor([19.9986], device='cuda:0')
Training epoch 3, validating ELBO:4246.92935899
Training epoch 4, ELBO:4090.

In [10]:
final_latent,gp_latent, gaussian_latent = model.batching_latent_samples(X=indices_all, Y=adata.X,
                                                                            batch_size=adata.X.shape[0])
np.savetxt(final_latent_file, final_latent, delimiter=",")
np.savetxt(gp_latent_file, gp_latent, delimiter=",")
np.savetxt(gaussian_latent_file, gaussian_latent, delimiter=",")